In [1]:
import sys
print(sys.executable)

D:\programfiles\anaconda3\python.exe


In [1]:
## Switch Capacitor using OTA for 1st Order DSM
import numpy as np
import matplotlib.pyplot as plt
import deltasigma as ds

## Calibration of test system
L = 1                        # Order of modulator
form = 'CIFB'                # Cascade of integrator feedback
fs = 220e3                   # Sampling frequency
M = 512                      # OSR
N = 16 * M                   # Simulation length / FFT points
fB = fs / 2 / M              # Bandwidth
cycles = 9                   # Number of sinusoids
fx = cycles * fs / N         # Test tone
A = 0.8                      # Signal amplitude

u = A * np.sin(2 * np.pi * fx / fs * np.arange(N))

## Design NTF
H = ds.synthesizeNTF(L, M)

## Pole-Zero Plot
fig1, ax1 = plt.subplots()
z, p, k = H.zeros, H.poles, H.gain

ax1.scatter(np.real(z), np.imag(z), s=80, marker='o',
            edgecolors='b', facecolors='c', linewidths=1.5, label='Zeros')
ax1.scatter(np.real(p), np.imag(p), s=100, marker='x',
            color='r', linewidths=2, label='Poles')

theta = np.linspace(0, 2 * np.pi, 1000)
unit_circle = np.exp(1j * theta)
ax1.plot(np.real(unit_circle), np.imag(unit_circle),
         '--k', linewidth=1.5, label='Unit Circle')

ax1.grid(True)
ax1.set_xlabel('Real Part')
ax1.set_ylabel('Imaginary Part')
ax1.set_title('Pole-Zero Plot of NTF')
ax1.legend(loc='best')
ax1.set_aspect('equal')
ax1.set_xlim([-1.5, 1.5])
ax1.set_ylim([-1.5, 1.5])

## Bode Plot
f = np.linspace(0, 0.5, N // 2 + 1)
z_vec = np.exp(2j * np.pi * f)
H_eval_bode = ds.evalTF(H, z_vec)

f_range = np.linspace(0, 0.5 / M, 1000)
z_range = np.exp(2j * np.pi * f_range)
H_eval = ds.evalTF(H, z_range)
rms_gain = np.sqrt(np.mean(np.abs(H_eval) ** 2))
sigma_H = 20 * np.log10(rms_gain)
print(f'RMS Gain (in dB): {sigma_H:.4f}')

fig2, ax2 = plt.subplots()
ax2.plot(f, ds.dbv(H_eval_bode), linewidth=2)
ax2.set_xlabel('Frequency f/fs')
ax2.set_ylabel('Magnitude (dB)')
ax2.set_title('Bode Plot of NTF')
ax2.grid(True)

## Realize SDM
swing = 0.5   # Amplifier output swing, Vp
umax = 0.9    # Scale system for inputs up to 0.9 of full-scale

a, g, b, c = ds.realizeNTF(H, form)
b[1:] = 0
ABCD = ds.stuffABCD(a, g, b, c, form)
ABCD = ds.scaleABCD(ABCD, nlev=2, f=0, xlim=swing, ymax=None, umax=umax)
a, g, b, c = ds.mapABCD(ABCD, form)

print('a coefficients:', a)
print('g coefficients:', g)
print('b coefficients:', b)
print('c coefficients:', c)

## Capacitor Sizing
Vdd = 1.8
Vref = Vdd
FullScale = Vdd
DR = 100 + 3          # Dynamic range in dB, plus 3-dB margin
k_B = 1.38e-23
T = 300
kT = k_B * T

v_n2 = (FullScale / 2) ** 2 / 2 / ds.undbp(DR)   # kT/(OSR*C1)
C1 = kT / (M * v_n2)
C2 = C1 / b[0] * FullScale / 1

print(f'Capacitor C1: {C1:.2e} F')
print(f'Capacitor C2: {C2:.2e} F')

## Rebuild ABCD and Simulate
ABCD_sim = ds.stuffABCD(a, g, b, c, form)
v, xn, xmax, y = ds.simulateDSM(u, ABCD_sim)

## Time-domain Plot
fig3, ax3 = plt.subplots()
tsamples = np.arange(2049)
ax3.stairs(u[:2049], np.arange(2050), baseline=None, label='u')
ax3.stairs(v[:2049], np.arange(2050), baseline=None, label='v')
ax3.set_xlim([0, 2048])
ax3.set_ylim([-1.2, 1.2])
ax3.set_xlabel('Time t/T')
ax3.set_ylabel('Amplitude')
ax3.legend()
ax3.set_title(r'1st Order $\Sigma\Delta$')
ax3.grid(True)

## Spectral Analysis, FFT
sq = np.abs(np.fft.fft(v))
f_fft = np.arange(N // 2) / N
FSR = 1.0

sq_hlf = sq[:N // 2] * 2 / N / FSR
sqdBFS = 20 * np.log10(np.where(sq_hlf == 0, 1e-15, sq_hlf))

sigbin = cycles        # 0-indexed: bin = cycles (since sigbin = 1+cycles in MATLAB 1-indexed)
noise = np.concatenate([sq_hlf[:sigbin], sq_hlf[sigbin + 1:]])
snr = 10 * np.log10(sq_hlf[sigbin] ** 2 / np.sum(noise ** 2))
print(f'SNR: {snr:.2f} dB')

## DFT Magnitude Plot
fig4, ax4 = plt.subplots()
ax4.plot(f_fft, sqdBFS, linewidth=2)
ax4.set_xlabel('Frequency f/fs')
ax4.set_ylabel('DFT Magnitude in dBFS')
ax4.grid(True)

## Normalized Spectrum Plot
sqFS = sq / (N / 2)
fig5, ax5 = plt.subplots()
ax5.plot(f_fft, ds.dbv(sqFS[:N // 2]), linewidth=2)
ax5.set_xlim([0, 0.06])
ax5.set_ylim([-150, 0])
ax5.grid(True)
ax5.set_ylabel('dBFS')
ax5.set_xlabel('f/fs')

## Windowed Plot (Hann)
specHW = np.fft.fft(v * ds.ds_hann(N)) / (N / 4)
fig6, ax6 = plt.subplots()
ax6.plot(f_fft, ds.dbv(np.abs(specHW[:N // 2])), linewidth=2)
ax6.set_xlim([0, 0.06])
ax6.set_ylim([-150, 0])
ax6.grid(True)
ax6.set_ylabel('dBFS')
ax6.set_xlabel('f/fs')

plt.show()

C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\deltasigma\__init__.py:16: SyntaxWarning: invalid escape sequence '\_'
  """


ModuleNotFoundError: No module named 'numpy.distutils'